# NB6g — `passive_ratio` recomputed correctly (CAMeL morphology) + the 6-vs-5 LOGO decision

**Single-pass notebook** — runs top-to-bottom in one Kaggle *Save Version* commit. No cell needs a
previous cell to be re-run; anything not already saved as a dataset (the CAMeLBERT-MSA embeddings) is
built inline, in order, before it is used.

**Goal.** In NB6f the winning 6-feature set
`{burstiness, ttr, quote_ratio, function_word_ratio, passive_ratio, compressibility}` scored
LOGO-worst **95.2**, but `passive_ratio` used the **word-shape fallback** (CAMeL was not installed),
which is unreliable for Arabic passive voice. Here we recompute **only** `passive_ratio` correctly —
CAMeL's morphological disambiguator reads the `vox` voice tag per verb (`p` = passive) — and isolate its effect.

| Config | dim | `passive_ratio` | role |
|---|---|---|---|
| neural-only | 768 | — | **anchor** → worst should reproduce ≈ 92.7 |
| winner-6 (fallback) | 774 | old word-shape | **anchor** → should reproduce ≈ 95.2 |
| winner-6 (correct) | 774 | CAMeL `vox=pas` | **the deciding number** |
| winner-5 (drop passive) | 773 | — | drop-to-5 option ≈ 94.7 |

If both anchors reproduce, the gap between the two winner-6 rows is due **only** to fixing
`passive_ratio`. **Rule:** correct-6 worst ≥ ~95.2 → keep 6; slides toward 94.7 → fallback was noise,
drop to the 5 analyzer-free features.

Everything except `passive_ratio` is *loaded* already-scaled from `vstat16` (byte-identical to the
earlier numbers). First run is long (CAMeL disambiguation on CPU + embedding extraction on GPU); to
make later versions fast, add this run's `/kaggle/working/*.parquet` outputs as input datasets and
point the paths below at them.

## 1 · Config

In [1]:
import os, numpy as np, pandas as pd

# Kaggle input paths — EDIT to your dataset names
P_DATASET = "/kaggle/input/notebooks/bahaaqassem/nb3-build-dataset/dataset.parquet"
P_VSTAT16 = "/kaggle/input/notebooks/bahaaqassem/ph2-nb6e-extract-11-features/vstat16_scaled.parquet"
# Embeddings + passive caches: point at datasets if you promoted a previous run's outputs; else the
# notebook builds them inline this run and writes them to /kaggle/working.
P_EMB     = "/kaggle/input/notebooks/bahaaqassem/nb6g-passive-ratio-decision/embeddings.parquet"
P_PASSIVE = "/kaggle/input/notebooks/bahaaqassem/nb6g-passive-ratio-decision/passive_ratio_camel_v2.parquet"
W_EMB     = "/kaggle/working/embeddings.parquet"
W_PASSIVE = "/kaggle/working/passive_ratio_camel_v2.parquet"
EXPECT_CACHED = True   # you have the cached files -> fail FAST if a path is wrong, instead of a silent 4h rebuild

FIVE_ANALYZER_FREE = ["burstiness", "ttr", "quote_ratio", "function_word_ratio", "compressibility"]
WINNER_6           = ["burstiness", "ttr", "quote_ratio", "function_word_ratio", "passive_ratio", "compressibility"]
FALLBACK_PASSIVE_COL = "passive_ratio"       # old fallback passive (already scaled) inside vstat16

EMB_MODEL   = "CAMeL-Lab/bert-base-arabic-camelbert-msa"   # MUST match NB6's encoder + pooling
GENERATORS  = ["deepseek", "sonnet", "qwen", "gemini", "gpt", "opus"]
LOGREG_KW   = dict(max_iter=2000, C=1.0, class_weight="balanced")   # n_jobs dropped (no effect + warns)
ORIGINAL_5 = ["targeted_ppl", "burstiness", "ttr", "entity_density", "discourse_coherence"]  # NB6f official-5
ANCHOR_NEURAL_WORST, ANCHOR_OFFICIAL5_WORST, ANCHOR_WINNER6_WORST, NOISE_FLOOR_PP = 92.7, 94.2, 95.2, 0.42
print("config loaded")

config loaded


## 2 · Install CAMeL Tools + MSA morphology DB

In [2]:
!pip install -q camel-tools
# The default MLEDisambiguator needs BOTH the MLE model AND the MSA morphology DB.
# morphology-db alone (as first tried) is NOT enough -> it looks for disambig_mle/calima-msa-r13/model.json
!camel_data -i disambig-mle-calima-msa-r13
!camel_data -i morphology-db-msa-r13
import os
p = os.path.expanduser("~/.camel_tools/data/disambig_mle/calima-msa-r13/model.json")
print("MLE model present:", os.path.exists(p))
# fail fast with a clear message if the data still is not there
if not os.path.exists(p):
    raise RuntimeError("CAMeL MLE data missing. Try:  !camel_data -i light   (bundles MSA disambig+morphology)")
print("camel-tools ready")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.7/125.7 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 91.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.3/122.3 kB 5.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sklearn-compat 0.1.5 requires scikit-learn<1.9,>=1.2, but you have scikit-learn 1.9.0 which is incompatible.
The following packages will be installed: 'morphology-db-msa-r13', 'disambig-mle-calima-msa-r13'
Extracting package 'morphology-db-msa-r13': 100%|█| 40.5M/40.5M [00:00<00:00, 47
Extracting package 'disambig-mle-calima-msa-r13': 100%|█| 88.7M/88.7M [00:00<00:
No new packages will be installed.
MLE model present: True
camel-tools ready


## 3 · Load + align dataset · vstat16

In [3]:
df = pd.read_parquet(P_DATASET)
if "article_id" in df.columns: df = df.set_index("article_id")
v16 = pd.read_parquet(P_VSTAT16)
if "article_id" in v16.columns: v16 = v16.set_index("article_id")

need = set(FIVE_ANALYZER_FREE) | {FALLBACK_PASSIVE_COL}
assert not (need - set(v16.columns)), f"vstat16 missing: {need - set(v16.columns)}"
df  = df.loc[df.index.intersection(v16.index)]
v16 = v16.loc[df.index]
assert (df.index == v16.index).all(), "df<->v16 alignment failed"
print(f"aligned articles: {len(df)} | label balance {df['label'].value_counts().to_dict()}")
print("splits:", df['split'].value_counts().to_dict())

aligned articles: 7101 | label balance {1: 3601, 0: 3500}
splits: {'train': 5363, 'test': 1093, 'val': 645}


## 4 · `passive_ratio` — correct definition

Arabic passive (المبني للمجهول) is marked by internal vowels (قُتِل vs قَتَل), **absent in
undiacritized news**. The word-shape fallback can't recover it; CAMeL's `MLEDisambiguator`
(`calima-msa-r13`) picks the most-likely analysis in context and exposes `vox` (`act`/`pas`) per verb.

`passive_ratio = (#tokens pos=verb & vox=pas) / (#tokens pos=verb)` — verb-normalised, so it measures
*voice preference*, not verb frequency. No verb → NaN → imputed with the **train-split median**.

*Thesis caveat to disclose:* even this is a statistical best-guess on undiacritized text, not gold
voice — but strictly better-grounded than word-shape. This notebook measures whether it moves LOGO.

## 5 · Compute correct `passive_ratio` (inline; reuses a cached dataset if present)

In [4]:
import os, re, multiprocessing as mp
from tqdm.auto import tqdm

CACHE = "/kaggle/working/passive_ratio_camel_v2.parquet"   # v2: v1 had the vox-tag bug (all zeros)

def _load_cache(*paths):
    for p in paths:
        if p and os.path.exists(p):
            d = pd.read_parquet(p)
            if "article_id" in d.columns: d = d.set_index("article_id")
            d = d.reindex(df.index)
            if d["passive_ratio_raw"].fillna(0).abs().sum() == 0:   # reject degenerate (buggy) cache
                print("cache is all-zero (old bug) -> recomputing"); return None
            return d
    return None

pr = _load_cache(P_PASSIVE, CACHE)
if pr is None and EXPECT_CACHED:
    raise FileNotFoundError("EXPECT_CACHED=True but no passive cache found.\n"
        f"  tried P_PASSIVE={P_PASSIVE}\n  tried CACHE={CACHE}\n"
        "Fix P_PASSIVE (copy the exact path from the Input pane) OR set EXPECT_CACHED=False to rebuild (~30min).")
if pr is not None:
    print("loaded cached passive_ratio_camel")
else:
    _SENT = re.compile(r'[.!\u061f?\n]+')
    PASSIVE_VOX = {"p", "pas", "passive"}          # CAMeL tags passive voice as 'p'
    def _init():
        global _mle, _tok
        from camel_tools.disambig.mle import MLEDisambiguator
        from camel_tools.tokenizers.word import simple_word_tokenize
        _mle = MLEDisambiguator.pretrained(); _tok = simple_word_tokenize
    def _one(text):
        verbs = passives = 0
        for sent in (s.strip() for s in _SENT.split(str(text)) if s.strip()):
            toks = _tok(sent)
            if not toks: continue
            for d in _mle.disambiguate(toks):
                if not d.analyses: continue
                a = d.analyses[0].analysis
                if a.get("pos") == "verb":
                    verbs += 1
                    if a.get("vox") in PASSIVE_VOX: passives += 1
        return (float("nan") if verbs == 0 else passives / verbs, verbs, passives)

    texts = df["text"].tolist(); results = None
    try:                                            # parallel across CPU cores (fork on Kaggle Linux)
        nproc = min(4, os.cpu_count() or 1)
        if nproc > 1:
            with mp.Pool(nproc, initializer=_init) as pool:
                results = list(tqdm(pool.imap(_one, texts, chunksize=16),
                                    total=len(texts), desc=f"passive x{nproc}"))
    except Exception as e:
        print("parallel failed -> sequential:", e); results = None
    if results is None:                             # safe fallback
        _init(); results = [_one(t) for t in tqdm(texts, desc="passive (seq)")]

    ratios = [r[0] for r in results]
    tot_v, tot_p = sum(r[1] for r in results), sum(r[2] for r in results)
    pr = pd.DataFrame({"passive_ratio_raw": ratios}, index=df.index)
    pr.reset_index().to_parquet(CACHE, index=False)
    print(f"DIAGNOSTIC: verbs={tot_v}  passive={tot_p}  passive share={100*tot_p/max(tot_v,1):.2f}%")
    print("saved ->", CACHE)

print("NaN (no verb):", int(pr['passive_ratio_raw'].isna().sum()),
      "| non-zero docs:", int((pr['passive_ratio_raw'].fillna(0)!=0).sum()))

loaded cached passive_ratio_camel
NaN (no verb): 0 | non-zero docs: 6313


## 6 · Scale correct passive (train-only) + assemble stat blocks

In [5]:
from sklearn.preprocessing import StandardScaler
train_mask = (df["split"] == "train").to_numpy()
raw = pr["passive_ratio_raw"].to_numpy(np.float64)
raw = np.where(np.isnan(raw), np.nanmedian(raw[train_mask]), raw)
sc = StandardScaler().fit(raw[train_mask].reshape(-1,1))
passive_correct_scaled = sc.transform(raw.reshape(-1,1)).ravel().astype(np.float32)

ai = df["label"].to_numpy() == 1
print(f"raw passive_ratio  human={raw[~ai].mean():.4f}  ai={raw[ai].mean():.4f}")

def build6(passive_col):
    cols = {f: v16[f].to_numpy(np.float32) for f in FIVE_ANALYZER_FREE}
    cols["passive_ratio"] = np.asarray(passive_col, np.float32).ravel()
    return np.column_stack([cols[f] for f in WINNER_6]).astype(np.float32)

STAT = {
    "winner6_fallback": build6(v16[FALLBACK_PASSIVE_COL].to_numpy(np.float32)),
    "winner6_correct" : build6(passive_correct_scaled),
    "five_drop"       : v16[FIVE_ANALYZER_FREE].to_numpy(np.float32),
}
if not (set(ORIGINAL_5) - set(v16.columns)):
    STAT["official5"] = v16[ORIGINAL_5].to_numpy(np.float32)   # NB6f 94.2 anchor
else:
    print("note: original-5 cols not all in vstat16 -> skipping official5 anchor")
for k,v in STAT.items(): print(k, v.shape)

raw passive_ratio  human=0.0436  ai=0.0333
winner6_fallback (7101, 6)
winner6_correct (7101, 6)
five_drop (7101, 5)
official5 (7101, 5)


## 7 · Embeddings (Vneural, 768) — load if saved, else build inline now

No prior notebook exported the CAMeLBERT-MSA vectors as a file (they were computed in-memory in NB6),
so on the first run this cell **builds** them: chunk to ≤510 content tokens, take each chunk's [CLS],
mean-pool to one 768-d vector — the documented Track-A pooling. It saves to `/kaggle/working` so you
can promote it to a dataset for fast future versions. **Must match NB6's pooling** or the 92.7 anchor
shifts; the anchor check in the verdict cell tells you if it does.

In [6]:
def _load_emb(*paths):
    for p in paths:
        if p and os.path.exists(p):
            e = pd.read_parquet(p)
            if "article_id" in e.columns: e = e.set_index("article_id")
            return e.reindex(df.index)
    return None

emb = _load_emb(P_EMB, W_EMB)
if emb is None and EXPECT_CACHED:
    raise FileNotFoundError("EXPECT_CACHED=True but no embeddings cache found.\n"
        f"  tried P_EMB={P_EMB}\n  tried W_EMB={W_EMB}\n"
        "Fix P_EMB (copy the exact path from the Input pane) OR set EXPECT_CACHED=False to rebuild (~4h).")
if emb is None:
    import torch
    from transformers import AutoTokenizer, AutoModel
    from tqdm.auto import tqdm
    tok = AutoTokenizer.from_pretrained(EMB_MODEL)
    model = AutoModel.from_pretrained(EMB_MODEL).eval()
    dev = "cuda" if torch.cuda.is_available() else "cpu"; model.to(dev)
    cls_id = tok.cls_token_id if tok.cls_token_id is not None else tok.convert_tokens_to_ids("[CLS]")
    sep_id = tok.sep_token_id if tok.sep_token_id is not None else tok.convert_tokens_to_ids("[SEP]")
    @torch.no_grad()
    def embed(text, max_ct=510, stride=460):
        # build [CLS]+chunk+[SEP] manually — avoids tokenizer.prepare_for_model (absent on this tokenizer)
        ids = tok(str(text), add_special_tokens=False)["input_ids"] or [tok.unk_token_id]
        chunks = [ids[i:i+max_ct] for i in range(0, len(ids), stride)] or [ids]
        vecs = []
        for c in chunks:
            inp = torch.tensor([[cls_id] + c + [sep_id]], device=dev)
            att = torch.ones_like(inp)
            out = model(input_ids=inp, attention_mask=att).last_hidden_state[:, 0, :]
            vecs.append(out.squeeze(0).cpu().numpy())
        return np.mean(vecs, axis=0).astype(np.float32)
    M = np.vstack([embed(t) for t in tqdm(df["text"].tolist(), desc="CAMeLBERT-MSA [CLS]")])
    emb = pd.DataFrame(M, index=df.index, columns=[f"n{i}" for i in range(768)])
    emb.reset_index().to_parquet(W_EMB, index=False)
    print("built + saved ->", W_EMB)
else:
    print("loaded cached embeddings")

NEURAL_COLS = [c for c in emb.columns if c != "article_id"]
assert len(NEURAL_COLS) == 768, f"expected 768 dims, got {len(NEURAL_COLS)}"
emb = emb.loc[df.index]
assert (emb.index == df.index).all(), "df<->emb alignment failed"
Xneural = emb[NEURAL_COLS].to_numpy(np.float32)
print("Vneural:", Xneural.shape)

loaded cached embeddings
Vneural: (7101, 768)


## 8 · LOGO harness (converged LogReg, pair-safe, test-split held-out generator)

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score
y   = df["label"].to_numpy()
gen = df["generator"].fillna("__human__").to_numpy()
tr_split = df["split"].isin(["train","val"]).to_numpy()
te_split = (df["split"] == "test").to_numpy()

def logo(Xstat, Xneural=None, scale=False):
    # Leave-one-generator-out, pair-safe (train+val minus held-out gen AI ; test-split gen AI + all test human).
    # scale=True standardizes the FULL fused vector per fold (fit on train rows) -> puts the raw 768 neural
    # dims and the stat dims on comparable magnitude, the likely NB6f operating point.
    X = Xstat if Xneural is None else np.hstack([Xneural, Xstat]).astype(np.float32)
    human = (y == 0); per = {}
    for g in GENERATORS:
        tr = tr_split & (human | (gen != g))
        te = te_split & (human | (gen == g))
        Xtr, Xte = X[tr], X[te]
        if scale:
            s = StandardScaler().fit(Xtr); Xtr = s.transform(Xtr); Xte = s.transform(Xte)
        clf = LogisticRegression(**LOGREG_KW).fit(Xtr, y[tr])
        per[g] = 100.0 * f1_score(y[te], clf.predict(Xte), average="macro")
    v = np.array(list(per.values())); return per, v.mean(), v.min()

for g in GENERATORS:
    print(f"  test AI held-out {g}: {int((te_split & (gen==g)).sum())}")
print("harness ready")

  test AI held-out deepseek: 135
  test AI held-out sonnet: 137
  test AI held-out qwen: 88
  test AI held-out gemini: 63
  test AI held-out gpt: 68
  test AI held-out opus: 63
harness ready


## 9 · Run all configs

In [8]:
MODES = [("raw", False), ("scaled", True)]
rows = []
for mode, sc in MODES:
    for name, X in STAT.items():                                   # stat-only (also #4 numbers)
        per,m,w = logo(X, None, scale=sc); rows.append((mode,"stat-only",name,X.shape[1],m,w,per))
    per,m,w = logo(np.zeros((len(y),0),np.float32), Xneural, scale=sc)   # neural-only anchor
    rows.append((mode,"hybrid","neural_only_768",768,m,w,per))
    for name, X in STAT.items():                                   # hybrid
        per,m,w = logo(X, Xneural, scale=sc); rows.append((mode,"hybrid",name,768+X.shape[1],m,w,per))

res = pd.DataFrame([(a,b,c,d,round(e,2),round(f,2)) for a,b,c,d,e,f,_ in rows],
                   columns=["mode","track","config","dim","LOGO_mean","LOGO_worst"])
import IPython.display as ipd; ipd.display(res[res.track=="hybrid"])

,mode,track,config,dim,LOGO_mean,LOGO_worst
4,raw,hybrid,neural_only_768,768,98.19,92.20
5,raw,hybrid,winner6_fallback,774,97.99,93.21
6,raw,hybrid,winner6_correct,774,97.87,92.71
7,raw,hybrid,five_drop,773,97.95,93.21
8,raw,hybrid,official5,773,98.25,93.12
13,scaled,hybrid,neural_only_768,768,98.78,95.09
14,scaled,hybrid,winner6_fallback,774,98.79,95.57
15,scaled,hybrid,winner6_correct,774,98.73,95.57
16,scaled,hybrid,five_drop,773,98.79,95.57
17,scaled,hybrid,official5,773,98.67,95.09


## 10 · Verdict

In [9]:
def get(mode,track,name,col):
    r = res[(res["mode"]==mode)&(res.track==track)&(res.config==name)]
    return None if r.empty else float(r[col].iloc[0])

print("="*66)
print("REPRODUCTION CHECK per mode (targets: neural 92.7 | official5 94.2 | winner6 95.2):")
score = {}
for mode,_ in MODES:
    nw = get(mode,"hybrid","neural_only_768","LOGO_worst")
    ow = get(mode,"hybrid","official5","LOGO_worst")
    fw = get(mode,"hybrid","winner6_fallback","LOGO_worst")
    print(f"  [{mode:6}] neural={nw:.2f}(d{nw-ANCHOR_NEURAL_WORST:+.1f})  "
          f"official5={ow:.2f}(d{ow-ANCHOR_OFFICIAL5_WORST:+.1f})  "
          f"winner6fb={fw:.2f}(d{fw-ANCHOR_WINNER6_WORST:+.1f})" if ow is not None else
          f"  [{mode:6}] neural={nw:.2f}(d{nw-ANCHOR_NEURAL_WORST:+.1f})  winner6fb={fw:.2f}(d{fw-ANCHOR_WINNER6_WORST:+.1f})")
    score[mode] = abs(fw-ANCHOR_WINNER6_WORST) + (abs(ow-ANCHOR_OFFICIAL5_WORST) if ow is not None else 0)

best = min(score, key=score.get)
print("="*66)
print(f"MODE THAT BEST REPRODUCES NB6f: '{best}'  -> reading the passive decision from it")
cw = get(best,"hybrid","winner6_correct","LOGO_worst")
dw = get(best,"hybrid","five_drop","LOGO_worst")
print(f"  winner-6 (correct passive) worst = {cw:.2f}")
print(f"  five   (drop passive)      worst = {dw:.2f}")
print(f"  delta (keep - drop)              = {cw-dw:+.2f}pp   (1-article noise ~ {NOISE_FLOOR_PP}pp)")
if   cw-dw >  NOISE_FLOOR_PP: print("  => KEEP passive_ratio (6 features).")
elif cw-dw < -NOISE_FLOOR_PP: print("  => DROP passive_ratio: use the 5 analyzer-free features.")
else: print("  => WITHIN NOISE: doesn't move the floor -> prefer the simpler analyzer-free 5.")
print("\nNOTE: trust this only if 'best' mode actually reproduced winner6fb~95.2 above; if neither mode")
print("      reproduces, the fusion operating point still differs from NB6f (check LogReg C / train composition).")

REPRODUCTION CHECK per mode (targets: neural 92.7 | official5 94.2 | winner6 95.2):
  [raw   ] neural=92.20(d-0.5)  official5=93.12(d-1.1)  winner6fb=93.21(d-2.0)
  [scaled] neural=95.09(d+2.4)  official5=95.09(d+0.9)  winner6fb=95.57(d+0.4)
MODE THAT BEST REPRODUCES NB6f: 'scaled'  -> reading the passive decision from it
  winner-6 (correct passive) worst = 95.57
  five   (drop passive)      worst = 95.57
  delta (keep - drop)              = +0.00pp   (1-article noise ~ 0.42pp)
  => WITHIN NOISE: doesn't move the floor -> prefer the simpler analyzer-free 5.

NOTE: trust this only if 'best' mode actually reproduced winner6fb~95.2 above; if neither mode
      reproduces, the fusion operating point still differs from NB6f (check LogReg C / train composition).
